# Saga Pattern & Dead Letter Queue

Distributed transaction handling.

In [ ]:
class SagaStep:
    def __init__(self, name, action, compensate):
        self.name = name
        self.action = action
        self.compensate = compensate

class SagaOrchestrator:
    def __init__(self, steps):
        self.steps = steps

    def execute(self):
        completed = []
        for step in self.steps:
            try:
                print(f"Executing: {step.name}")
                step.action()
                completed.append(step)
            except Exception as e:
                print(f"Failed at {step.name}: {e}")
                self._rollback(completed)
                return False
        return True

    def _rollback(self, completed):
        for step in reversed(completed):
            print(f"Compensating: {step.name}")
            step.compensate()

# Example: Order Saga
def reserve_inventory(): print("  Inventory reserved")
def release_inventory(): print("  Inventory released")
def charge_payment(): raise Exception("Payment failed")
def refund_payment(): print("  Payment refunded")

saga = SagaOrchestrator([
    SagaStep("Reserve Inventory", reserve_inventory, release_inventory),
    SagaStep("Charge Payment", charge_payment, refund_payment),
])

saga.execute()